## Pengaturan lokasi
Ubah `PROJECT_ROOT` dan `VIDEO_DIR` di sel konfigurasi sebelum menjalankan. Gunakan workspace yang sama pada kedua notebook. Jalankan notebook ekstraksi terlebih dahulu; inventaris metadata dibuat otomatis jika belum ada CSV. Gunakan workspace baru jika mengganti dataset agar cache tidak tercampur.


# 01 — Pose Extraction: WL-BISINDO → 65-Keypoint Sequences

Notebook ini mengekstraksi setiap video RGB WL-BISINDO menjadi sequence pose mentah berukuran **(T, 65, 2)**, dengan:

- `T`: jumlah frame asli video;
- `65`: 21 keypoint tangan kiri + 21 tangan kanan + 11 kepala/wajah + 12 tubuh bagian atas;
- `2`: koordinat `(x, y)` ternormalisasi dari MediaPipe.

Standardisasi temporal menjadi **64 frame** dilakukan pada notebook training dan preprocessing (`02_resgcn_loso_e0_e7.ipynb`) agar raw pose tetap mempertahankan panjang sequence aslinya.

Notebook bersifat **resume-safe**: pose yang sudah berhasil diekstrak tidak diproses ulang, sedangkan laporan quality control dibangun dari seluruh output yang tersedia.

## Dataset

Penelitian ini menggunakan **WL-BISINDO (Word-Level BISINDO)** yang diperkenalkan oleh **Grace Oktaviani Kindy, Glenn Leonali, dan Henry Lucky** dari BINUS University dalam paper:

> *Word-Level BISINDO: A Novel Video Indonesian Sign Language Dataset and Baseline Methods*  
> Procedia Computer Science, Vol. 269, 2025, pp. 249–258  
> DOI: https://doi.org/10.1016/j.procs.2025.08.277

WL-BISINDO merupakan dataset video **Bahasa Isyarat Indonesia (BISINDO)** pada tingkat kata dengan fokus pada **varian regional Banten**. Dataset berisi **1.600 video RGB**, **32 isolated signs**, dan **5 signers**.

Dataset tersedia secara publik melalui Kaggle:

https://www.kaggle.com/datasets/glennleonali/wl-bisindo

Lisensi dataset: **CC BY-NC 4.0** (penggunaan non-komersial dengan atribusi).

## Input yang Diharapkan

```text
<PROJECT_ROOT>/data/
├── videos/
│   ├── signer0_label0_sample0.mp4
│   └── ...
└── splits/
    └── *.csv
```

Setiap file metadata split harus memiliki kolom `video_id`.

## Output Utama

```text
<PROJECT_ROOT>/data/
├── data_pose_raw/*.npz
├── configs/joints65.yaml
├── configs/extraction_environment.json
├── reports/pose_qc.csv
├── reports/pose_qc_by_signer.csv
├── reports/pose_qc_by_label.csv
└── reports/pose_extraction_status.csv
```

Jalankan cell secara berurutan dari atas ke bawah.


## 0. Cara Menjalankan

1. Pastikan folder `videos/` dan `splits/` tersedia pada Google Drive.
2. Jalankan seluruh cell secara berurutan.
3. Jika runtime terputus, jalankan ulang notebook. File `.npz` yang sudah ada akan dilewati.
4. Setelah ekstraksi, selalu periksa `reports/pose_qc.csv` dan ringkasan per signer sebelum masuk ke training.


In [ ]:
# KONFIGURASI PENGGUNA: samakan lokasi ini pada kedua notebook.
from pathlib import Path
import os

# Aktifkan hanya jika input/output memang disimpan di Google Drive.
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Mount Google Drive hanya tersedia di Colab.") from exc
    drive.mount("/content/drive", force_remount=False)

# Bisa diganti langsung atau lewat environment variable WLBISINDO_ROOT.
# Contoh Drive: /content/drive/MyDrive/wl-bisindo-workspace
# Contoh Windows: D:/penelitian/wl-bisindo-workspace
PROJECT_ROOT = Path(os.environ.get("WLBISINDO_ROOT", "~/wl-bisindo-workspace")).expanduser().resolve()
# Ubah VIDEO_DIR bila video berada di lokasi lain (subfolder diperbolehkan).
VIDEO_DIR = PROJECT_ROOT / "data" / "videos"
DATA_DIR = PROJECT_ROOT / "data"
print("Workspace penelitian:", PROJECT_ROOT)
if not USE_GOOGLE_DRIVE:
    print("Di Colab, penyimpanan lokal dapat hilang saat runtime berakhir. Salin hasil atau pilih Drive.")

SPLITS_DIR = DATA_DIR / "splits"
OUT_POSE_DIR = DATA_DIR / "data_pose_raw"
REPORT_DIR = DATA_DIR / "reports"
CONFIG_DIR = DATA_DIR / "configs"
MODEL_DIR = DATA_DIR / "mp_models"

EXPECTED_NUM_VIDEOS = 1600
EXPECTED_NUM_CLASSES = 32
EXPECTED_SIGNERS = {0, 1, 2, 3, 4}

for p in [OUT_POSE_DIR, REPORT_DIR, CONFIG_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("VIDEO_DIR    :", VIDEO_DIR)
print("SPLITS_DIR   :", SPLITS_DIR)
print("OUT_POSE_DIR :", OUT_POSE_DIR)


## 1. Instalasi Dependency


In [ ]:
%pip -q install mediapipe opencv-python PyYAML


## 2. Import dan Pencatatan Environment


In [ ]:
import gc
import csv
import json
import re
import sys
import platform
import importlib.metadata as importlib_metadata

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from IPython.display import display

def package_version(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return None

environment = {
    "created_at": __import__("datetime").datetime.now().isoformat(timespec="seconds"),
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {
        "numpy": package_version("numpy"),
        "pandas": package_version("pandas"),
        "opencv-python": package_version("opencv-python"),
        "mediapipe": package_version("mediapipe"),
        "PyYAML": package_version("PyYAML"),
        "tqdm": package_version("tqdm"),
    },
}

env_path = CONFIG_DIR / "extraction_environment.json"
with open(env_path, "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2)

print("Environment recorded:", env_path)
display(pd.DataFrame([environment["packages"]]))


## 3. MediaPipe Model Assets


In [ ]:
import urllib.request

POSE_TASK = MODEL_DIR / "pose_landmarker_full.task"
HAND_TASK = MODEL_DIR / "hand_landmarker.task"

# Primary URLs (latest)
URLS = {
    POSE_TASK: "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
    HAND_TASK: "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
}

# Fallback URLs (if 'latest' ever fails)
FALLBACK = {
    POSE_TASK: "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
    HAND_TASK: "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
}

def download_if_missing(path: Path, url: str, fallback_url: str):
    if path.exists() and path.stat().st_size > 0:
        print(f"OK: {path.name} already exists.")
        return
    try:
        print(f"Downloading {path.name} ...")
        urllib.request.urlretrieve(url, str(path))
        print("Saved:", path)
    except Exception as e:
        print("Primary download failed:", e)
        print("Trying fallback ...")
        urllib.request.urlretrieve(fallback_url, str(path))
        print("Saved:", path)

download_if_missing(POSE_TASK, URLS[POSE_TASK], FALLBACK[POSE_TASK])
download_if_missing(HAND_TASK, URLS[HAND_TASK], FALLBACK[HAND_TASK])


## 4. Definisi 65 Keypoint


In [ ]:
import yaml

JOINT_SPEC = {
  "version": "WLBISINDO-65-v1",
  "nodes": []
}

for i in range(21):
    JOINT_SPEC["nodes"].append({"idx": i, "name": f"LH_{i}", "source": "hand_left", "mp_index": i})
for i in range(21):
    JOINT_SPEC["nodes"].append({"idx": 21+i, "name": f"RH_{i}", "source": "hand_right", "mp_index": i})

head_pose = [
    ("NOSE", 0),
    ("LEFT_EYE_INNER", 1), ("LEFT_EYE", 2), ("LEFT_EYE_OUTER", 3),
    ("RIGHT_EYE_INNER", 4), ("RIGHT_EYE", 5), ("RIGHT_EYE_OUTER", 6),
    ("LEFT_EAR", 7), ("RIGHT_EAR", 8),
    ("MOUTH_LEFT", 9), ("MOUTH_RIGHT", 10),
]
base = 42
for j, (nm, mp_idx) in enumerate(head_pose):
    JOINT_SPEC["nodes"].append({"idx": base+j, "name": nm, "source": "pose", "mp_index": mp_idx})

upper_body = [
    ("NECK", None),
    ("LEFT_SHOULDER", 11), ("RIGHT_SHOULDER", 12),
    ("LEFT_ELBOW", 13), ("RIGHT_ELBOW", 14),
    ("LEFT_WRIST", 15), ("RIGHT_WRIST", 16),
    ("LEFT_HIP", 23), ("RIGHT_HIP", 24),
    ("MID_HIP", None),
    ("LEFT_INDEX", 19), ("RIGHT_INDEX", 20),
]
base = 53
for j, (nm, mp_idx) in enumerate(upper_body):
    JOINT_SPEC["nodes"].append({"idx": base+j, "name": nm, "source": "pose" if mp_idx is not None else "derived", "mp_index": mp_idx})

spec_path = CONFIG_DIR / "joints65.yaml"
with open(spec_path, "w") as f:
    yaml.safe_dump(JOINT_SPEC, f, sort_keys=False)

print("Saved:", spec_path, "| total nodes =", len(JOINT_SPEC["nodes"]))


## 5. Metadata Dataset dan Integrity Check


In [ ]:
# Daftar video baru dibuat dari nama file, tanpa metadata pribadi penulis.
# Ini merekonstruksi inventaris input, bukan menjamin split historis identik.
video_paths = {}
pattern_inventory = re.compile(r"signer(\d+)_label(\d+)_sample(\d+)", re.IGNORECASE)
for video in sorted(VIDEO_DIR.rglob("*")):
    if video.is_file() and video.suffix.lower() == ".mp4":
        if not pattern_inventory.fullmatch(video.stem):
            raise ValueError(f"Nama video tidak sesuai pola: {video.name}")
        if video.stem in video_paths:
            raise ValueError(f"ID video duplikat: {video.stem}")
        video_paths[video.stem] = video
if not video_paths:
    raise FileNotFoundError(f"Tidak ada video MP4 di {VIDEO_DIR}; atur VIDEO_DIR terlebih dahulu.")
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
# CSV yang sudah tersedia tetap dipakai. Untuk dataset baru, gunakan workspace baru.
if not list(SPLITS_DIR.rglob("*.csv")):
    pd.DataFrame({"video_id": sorted(video_paths)}).to_csv(SPLITS_DIR / "video_inventory.csv", index=False)
    print("Inventaris video dibuat:", SPLITS_DIR / "video_inventory.csv")

def list_split_csvs(splits_dir: Path):
    return sorted([p for p in splits_dir.rglob("*.csv") if p.is_file()])

csvs = list_split_csvs(SPLITS_DIR)
if not csvs:
    raise FileNotFoundError(f"Tidak ditemukan CSV metadata di {SPLITS_DIR}")

dfs = []
for p in csvs:
    d = pd.read_csv(p)
    if "video_id" in d.columns:
        dfs.append(d[["video_id"]])

if not dfs:
    raise RuntimeError("Tidak ada CSV yang memiliki kolom 'video_id'.")

all_ids = (
    pd.concat(dfs, axis=0)["video_id"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values()
    .tolist()
)

pattern = re.compile(r"signer(\d+)_label(\d+)_sample(\d+)", re.IGNORECASE)

def parse_video_id(vid):
    m = pattern.fullmatch(str(vid))
    if not m:
        return None
    return int(m.group(1)), int(m.group(2)), int(m.group(3))

rows = []
unparsed = []
for vid in all_ids:
    parsed = parse_video_id(vid)
    if parsed is None:
        unparsed.append(vid)
        continue
    signer_id, label, sample_id = parsed
    rows.append({
        "video_id": vid,
        "signer_id": signer_id,
        "label": label,
        "sample_id": sample_id,
    })

df_vid = pd.DataFrame(rows).sort_values(
    ["signer_id", "label", "sample_id"]
).reset_index(drop=True)

if unparsed:
    raise ValueError(
        f"{len(unparsed)} video_id tidak mengikuti pola "
        "'signerN_labelK_sampleI'. Contoh: {unparsed[:5]}"
    )

# Sanity checks agar input yang diproses sesuai dataset penelitian.
if len(df_vid) != EXPECTED_NUM_VIDEOS:
    raise ValueError(
        f"Jumlah video tidak sesuai: ditemukan {len(df_vid)}, "
        f"ekspektasi {EXPECTED_NUM_VIDEOS}."
    )
if df_vid["label"].nunique() != EXPECTED_NUM_CLASSES:
    raise ValueError(
        f"Jumlah kelas tidak sesuai: ditemukan {df_vid['label'].nunique()}, "
        f"ekspektasi {EXPECTED_NUM_CLASSES}."
    )
if set(df_vid["signer_id"].unique()) != EXPECTED_SIGNERS:
    raise ValueError(
        f"Signer tidak sesuai: ditemukan {sorted(df_vid['signer_id'].unique())}, "
        f"ekspektasi {sorted(EXPECTED_SIGNERS)}."
    )

def resolve_video_path(video_id: str) -> Path:
    return video_paths.get(video_id, VIDEO_DIR / f"{video_id}.mp4")

missing_video_files = [vid for vid in all_ids if not resolve_video_path(vid).exists()]
print("Split CSVs      :", len(csvs))
print("Unique videos   :", len(df_vid))
print("Unique labels   :", df_vid["label"].nunique())
print("Signers         :", sorted(df_vid["signer_id"].unique()))
print("Missing MP4     :", len(missing_video_files))

if missing_video_files:
    print("Contoh video yang tidak ditemukan:", missing_video_files[:5])

display(df_vid.head())


## 6. Inisialisasi Pose dan Hand Landmarker


In [ ]:
# Create PoseLandmarker
pose_base = python.BaseOptions(model_asset_path=str(POSE_TASK))
pose_opts = vision.PoseLandmarkerOptions(
    base_options=pose_base,
    running_mode=vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False,
)
pose_detector = vision.PoseLandmarker.create_from_options(pose_opts)

# Create HandLandmarker
hand_base = python.BaseOptions(model_asset_path=str(HAND_TASK))
hand_opts = vision.HandLandmarkerOptions(
    base_options=hand_base,
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
hand_detector = vision.HandLandmarker.create_from_options(hand_opts)

print("Detectors ready.")


## 7. Fungsi Representasi Pose

Implementasi berikut mempertahankan representasi yang digunakan pada eksperimen: tangan kiri, tangan kanan, 11 titik kepala/wajah, dan 12 titik tubuh bagian atas disusun menjadi 65 node. Keypoint yang tidak terdeteksi disimpan sebagai `(0, 0)`.


In [ ]:
POSE_IDX = {
    "NOSE": 0,
    "LEFT_EYE_INNER": 1, "LEFT_EYE": 2, "LEFT_EYE_OUTER": 3,
    "RIGHT_EYE_INNER": 4, "RIGHT_EYE": 5, "RIGHT_EYE_OUTER": 6,
    "LEFT_EAR": 7, "RIGHT_EAR": 8,
    "MOUTH_LEFT": 9, "MOUTH_RIGHT": 10,
    "LEFT_SHOULDER": 11, "RIGHT_SHOULDER": 12,
    "LEFT_ELBOW": 13, "RIGHT_ELBOW": 14,
    "LEFT_WRIST": 15, "RIGHT_WRIST": 16,
    "LEFT_INDEX": 19, "RIGHT_INDEX": 20,
    "LEFT_HIP": 23, "RIGHT_HIP": 24,
}

def _zeros(n):
    return np.zeros((n,2), dtype=np.float32)

def _pose_xy(pose_list, idx):
    if pose_list is None:
        return np.array([0.0, 0.0], dtype=np.float32)
    lm = pose_list[idx]
    return np.array([lm.x, lm.y], dtype=np.float32)

def _hand_xy(hand_list):
    # hand_list: List[NormalizedLandmark] length 21
    if hand_list is None:
        return _zeros(21)
    pts = np.zeros((21,2), dtype=np.float32)
    for i, lm in enumerate(hand_list):
        pts[i,0] = lm.x
        pts[i,1] = lm.y
    return pts

def extract_65(pose_list, left_hand, right_hand):
    lh = _hand_xy(left_hand)
    rh = _hand_xy(right_hand)

    head = np.stack([
        _pose_xy(pose_list, POSE_IDX["NOSE"]),
        _pose_xy(pose_list, POSE_IDX["LEFT_EYE_INNER"]),
        _pose_xy(pose_list, POSE_IDX["LEFT_EYE"]),
        _pose_xy(pose_list, POSE_IDX["LEFT_EYE_OUTER"]),
        _pose_xy(pose_list, POSE_IDX["RIGHT_EYE_INNER"]),
        _pose_xy(pose_list, POSE_IDX["RIGHT_EYE"]),
        _pose_xy(pose_list, POSE_IDX["RIGHT_EYE_OUTER"]),
        _pose_xy(pose_list, POSE_IDX["LEFT_EAR"]),
        _pose_xy(pose_list, POSE_IDX["RIGHT_EAR"]),
        _pose_xy(pose_list, POSE_IDX["MOUTH_LEFT"]),
        _pose_xy(pose_list, POSE_IDX["MOUTH_RIGHT"]),
    ], axis=0).astype(np.float32)

    l_sh = _pose_xy(pose_list, POSE_IDX["LEFT_SHOULDER"])
    r_sh = _pose_xy(pose_list, POSE_IDX["RIGHT_SHOULDER"])
    neck = (l_sh + r_sh) / 2.0

    l_hip = _pose_xy(pose_list, POSE_IDX["LEFT_HIP"])
    r_hip = _pose_xy(pose_list, POSE_IDX["RIGHT_HIP"])
    midhip = (l_hip + r_hip) / 2.0

    upper = np.stack([
        neck,
        l_sh, r_sh,
        _pose_xy(pose_list, POSE_IDX["LEFT_ELBOW"]), _pose_xy(pose_list, POSE_IDX["RIGHT_ELBOW"]),
        _pose_xy(pose_list, POSE_IDX["LEFT_WRIST"]), _pose_xy(pose_list, POSE_IDX["RIGHT_WRIST"]),
        l_hip, r_hip,
        midhip,
        _pose_xy(pose_list, POSE_IDX["LEFT_INDEX"]), _pose_xy(pose_list, POSE_IDX["RIGHT_INDEX"]),
    ], axis=0).astype(np.float32)

    X = np.concatenate([lh, rh, head, upper], axis=0)
    assert X.shape == (65,2)
    return X


## 8. Ekstraksi Seluruh Video

Ekstraksi dilakukan pada seluruh frame asli video. Output setiap video disimpan terpisah sebagai `.npz` dengan key `X_raw`. Mekanisme resume dan atomic save digunakan agar sesi Colab yang terputus tidak meninggalkan file setengah jadi.


In [ ]:
QC_PATH = REPORT_DIR / "pose_qc.csv"
STATUS_PATH = REPORT_DIR / "pose_extraction_status.csv"

BATCH_SIZE = 200
SKIP_IF_EXISTS = True

STATUS_COLUMNS = ["video_id", "status", "T", "path", "message"]

def atomic_write_csv(df, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def save_status_updates(rows):
    if not rows:
        return
    new = pd.DataFrame(rows)
    if STATUS_PATH.exists():
        old = pd.read_csv(STATUS_PATH)
        merged = pd.concat([old, new], ignore_index=True)
    else:
        merged = new
    merged = (
        merged.drop_duplicates(subset=["video_id"], keep="last")
        .sort_values("video_id")
        .reset_index(drop=True)
    )
    for col in STATUS_COLUMNS:
        if col not in merged.columns:
            merged[col] = ""
    atomic_write_csv(merged[STATUS_COLUMNS], STATUS_PATH)

def compute_missing_rates(X):
    if X.ndim != 3 or X.shape[1:] != (65, 2):
        raise ValueError(f"Expected (T,65,2), got {X.shape}")
    miss = (np.abs(X).sum(axis=-1) == 0.0)  # (T,65)
    per_joint = miss.mean(axis=0)
    return {
        "missing_total": float(per_joint.mean()),
        "missing_lh": float(per_joint[0:21].mean()),
        "missing_rh": float(per_joint[21:42].mean()),
        "missing_head": float(per_joint[42:53].mean()),
        "missing_upper": float(per_joint[53:65].mean()),
    }

def get_todo(video_ids):
    todo = []
    for vid in video_ids:
        out_path = OUT_POSE_DIR / f"{vid}.npz"
        if not (SKIP_IF_EXISTS and out_path.exists()):
            todo.append(vid)
    return todo

todo = get_todo(all_ids)
print(f"Remaining to extract: {len(todo)} / {len(all_ids)}")

batch_num = 0
global_ts_ms = 0  # MediaPipe VIDEO mode requires monotonically increasing timestamps.

while todo:
    batch_num += 1
    batch = todo[:BATCH_SIZE]
    todo = todo[BATCH_SIZE:]

    print(f"\nBatch {batch_num}: {len(batch)} videos")
    status_rows = []

    for vid in tqdm(batch, desc=f"Batch {batch_num}", leave=False):
        out_path = OUT_POSE_DIR / f"{vid}.npz"
        video_path = resolve_video_path(vid)

        if SKIP_IF_EXISTS and out_path.exists():
            try:
                X = np.load(out_path, allow_pickle=False)["X_raw"]
                status_rows.append({
                    "video_id": vid,
                    "status": "exists",
                    "T": int(X.shape[0]),
                    "path": str(video_path),
                    "message": "",
                })
            except Exception as e:
                status_rows.append({
                    "video_id": vid,
                    "status": "corrupt_existing_output",
                    "T": 0,
                    "path": str(video_path),
                    "message": repr(e),
                })
            continue

        if not video_path.exists():
            status_rows.append({
                "video_id": vid,
                "status": "missing_file",
                "T": 0,
                "path": str(video_path),
                "message": "Source MP4 not found",
            })
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            status_rows.append({
                "video_id": vid,
                "status": "cannot_open",
                "T": 0,
                "path": str(video_path),
                "message": "cv2.VideoCapture failed",
            })
            cap.release()
            continue

        try:
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps is None or fps <= 1e-6 or np.isnan(fps):
                fps = 30.0
            dt_ms = max(1, int(round(1000.0 / fps)))

            X_frames = []

            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(
                    image_format=mp.ImageFormat.SRGB,
                    data=frame_rgb,
                )

                ts = global_ts_ms
                global_ts_ms += dt_ms

                pose_res = pose_detector.detect_for_video(mp_image, ts)
                pose_list = (
                    pose_res.pose_landmarks[0]
                    if pose_res.pose_landmarks
                    else None
                )

                hand_res = hand_detector.detect_for_video(mp_image, ts)
                left_hand = None
                right_hand = None

                if hand_res.hand_landmarks and hand_res.handedness:
                    for i in range(len(hand_res.hand_landmarks)):
                        label = hand_res.handedness[i][0].category_name
                        lms = hand_res.hand_landmarks[i]

                        if label == "Left":
                            left_hand = lms
                        elif label == "Right":
                            right_hand = lms

                X_frames.append(
                    extract_65(pose_list, left_hand, right_hand)
                )

        except Exception as e:
            status_rows.append({
                "video_id": vid,
                "status": "error",
                "T": 0,
                "path": str(video_path),
                "message": repr(e),
            })
            continue

        finally:
            cap.release()
            # Small gap between videos while retaining monotonic timestamps.
            global_ts_ms += 1000

        if len(X_frames) == 0:
            status_rows.append({
                "video_id": vid,
                "status": "empty_video",
                "T": 0,
                "path": str(video_path),
                "message": "No readable frames",
            })
            continue

        X_raw = np.stack(X_frames, axis=0).astype(np.float32)
        if X_raw.shape[1:] != (65, 2):
            raise AssertionError(f"{vid}: unexpected shape {X_raw.shape}")

        # Atomic save: partial files are not left behind if the runtime stops mid-write.
        tmp_path = out_path.with_name(out_path.stem + ".tmp.npz")
        np.savez_compressed(tmp_path, X_raw=X_raw)
        tmp_path.replace(out_path)

        status_rows.append({
            "video_id": vid,
            "status": "ok",
            "T": int(X_raw.shape[0]),
            "path": str(video_path),
            "message": "",
        })

        del X_frames, X_raw
        gc.collect()

    save_status_updates(status_rows)
    print("Status updated:", STATUS_PATH)

print("\nExtraction pass finished.")
print("Pose outputs:", OUT_POSE_DIR)


## 9. Quality Control Hasil Ekstraksi

QC dibangun ulang dari seluruh `video_id` yang diharapkan. Dengan demikian, laporan tidak menjadi duplikat walaupun notebook dijalankan berkali-kali. Missing keypoint didefinisikan sebagai koordinat `(0, 0)`.


In [ ]:
# ------------------------------------------------------------------
# REBUILD FULL QC DETERMINISTICALLY FROM ALL EXPECTED VIDEO IDs
# ------------------------------------------------------------------
if STATUS_PATH.exists():
    status_df = pd.read_csv(STATUS_PATH)
    status_map = status_df.set_index("video_id").to_dict("index")
else:
    status_map = {}

qc_rows = []

for vid in tqdm(all_ids, desc="Build full QC"):
    out_path = OUT_POSE_DIR / f"{vid}.npz"
    video_path = resolve_video_path(vid)
    meta = status_map.get(vid, {})

    row = {
        "video_id": vid,
        "status": meta.get("status", ""),
        "T": 0,
        "path": str(video_path),
        "missing_total": np.nan,
        "missing_lh": np.nan,
        "missing_rh": np.nan,
        "missing_head": np.nan,
        "missing_upper": np.nan,
    }

    if out_path.exists():
        try:
            X = np.load(out_path, allow_pickle=False)["X_raw"].astype(np.float32)
            if X.ndim != 3 or X.shape[1:] != (65, 2):
                raise ValueError(f"Unexpected shape {X.shape}")
            row["status"] = row["status"] or "exists"
            row["T"] = int(X.shape[0])
            row.update(compute_missing_rates(X))
        except Exception as e:
            row["status"] = "corrupt_pose_output"
            print(f"[WARN] {vid}: {e}")
    else:
        if not video_path.exists():
            row["status"] = row["status"] or "missing_file"
        else:
            row["status"] = row["status"] or "missing_pose_output"

    qc_rows.append(row)

qc = pd.DataFrame(qc_rows)
atomic_write_csv(qc, QC_PATH)

# Add signer/label metadata for summaries.
qc_meta = qc.merge(
    df_vid[["video_id", "signer_id", "label", "sample_id"]],
    on="video_id",
    how="left",
)

ok_mask = qc_meta["T"].gt(0) & qc_meta["missing_total"].notna()
qc_ok = qc_meta[ok_mask].copy()

signer_stats = (
    qc_ok.groupby("signer_id")
    .agg(
        n=("video_id", "count"),
        mean_T=("T", "mean"),
        min_T=("T", "min"),
        max_T=("T", "max"),
        mean_missing_total=("missing_total", "mean"),
        mean_missing_lh=("missing_lh", "mean"),
        mean_missing_rh=("missing_rh", "mean"),
        mean_missing_head=("missing_head", "mean"),
        mean_missing_upper=("missing_upper", "mean"),
    )
    .reset_index()
)

label_stats = (
    qc_ok.groupby("label")
    .agg(
        n=("video_id", "count"),
        mean_T=("T", "mean"),
        pct_T_lt_64=("T", lambda s: float((s < 64).mean())),
        mean_missing_total=("missing_total", "mean"),
        mean_missing_lh=("missing_lh", "mean"),
        mean_missing_rh=("missing_rh", "mean"),
    )
    .reset_index()
)

atomic_write_csv(signer_stats, REPORT_DIR / "pose_qc_by_signer.csv")
atomic_write_csv(label_stats, REPORT_DIR / "pose_qc_by_label.csv")

print("QC saved:", QC_PATH)
print("\nStatus counts:")
display(qc["status"].value_counts(dropna=False).rename("n").to_frame())

print("\nSequence length:")
display(qc_ok["T"].describe().to_frame("T"))

print("\nMissing-keypoint summary:")
display(
    qc_ok[
        ["missing_total", "missing_lh", "missing_rh", "missing_head", "missing_upper"]
    ].describe()
)

print("\nBy signer:")
display(signer_stats)


## 10. Cleanup


In [ ]:
# Release MediaPipe resources explicitly.
try:
    pose_detector.close()
except Exception:
    pass

try:
    hand_detector.close()
except Exception:
    pass

print("Detectors closed.")
